# Notebook recupere - Entrainement et evaluation du modele H2S

Ce notebook reconstruit le pipeline complet d'entrainement du modele IA H2S :

- chargement de `DATASET01.csv` ;
- nettoyage et controle qualite ;
- creation de la concentration fusionnee `C_H2S_fusion_ppm` ;
- entrainement et evaluation du modele Random Forest ;
- entrainement et evaluation du modele LSTM ;
- calcul des metriques : accuracy, balanced accuracy, precision, recall, F1-score, matrice de confusion, MAE, RMSE et R2 ;
- sauvegarde optionnelle des modeles.

> Remarque importante : le prototype embarque actuel utilise un seul capteur MQ-136. Le dataset historique contient quatre colonnes capteurs H2S ; pour l'entrainement, on les fusionne par moyenne. En deploiement capteur unique, `C_H2S_fusion_ppm = h2s_ppm`.


## 1. Installation et imports

Dans Google Colab, si une bibliotheque manque, executer la ligne d'installation affichee dans l'erreur. En local, utiliser l'environnement Python du projet.


In [ ]:
import os
import json
import random
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_recall_fscore_support,
    r2_score,
)
from sklearn.model_selection import train_test_split, learning_curve, StratifiedKFold
from sklearn.utils import resample
import joblib

try:
    from IPython.display import display
except Exception:
    display = print

try:
    import tensorflow as tf
    from tensorflow.keras import Sequential
    from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
except Exception as exc:
    tf = None
    print("TensorFlow indisponible :", exc)
    print("Dans Colab, executer par exemple : !pip install tensorflow")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
if tf is not None:
    tf.random.set_seed(SEED)

SMOKE_TEST = os.environ.get("NOTEBOOK_SMOKE_TEST", "0") == "1"
print("Mode test rapide :", SMOKE_TEST)


## 2. Parametres du projet

Pour obtenir les metriques finales, laisser `RUN_RF_TRAINING=True` et `RUN_LSTM_TRAINING=True`. Le mode test rapide est active seulement si la variable d'environnement `NOTEBOOK_SMOKE_TEST=1` est presente.


In [ ]:
# Colonnes du dataset historique.
TIME_COL = "Time[sec]"
SENSOR_COLS = ["Sensor1[ppm]", "Sensor2[ppm]", "Sensor3[ppm]", "Sensor4[ppm]"]
TEMP_COL = "Temperature[C]"
HUM_COL = "Humidity[%]"
TRUE_CONC_COL = "True_concentration[ppm]"
DATASET_LABEL_COL = "Labels"
FUSION_COL = "C_H2S_fusion_ppm"
RISK_CLASS_COL = "risk_class"

# Seuils H2S utilises dans le projet.
H2S_NORMAL_MAX = 10.0
H2S_MODERATE_MAX = 20.0
RISK_LABELS = {0: "NORMAL", 1: "MODERE", 2: "DANGEREUX"}

# Options d'entrainement.
RUN_RF_TRAINING = True
RUN_LSTM_TRAINING = True
RUN_LEARNING_CURVE = not SMOKE_TEST
SAVE_MODELS = False  # Mettre True seulement apres validation des resultats.

# Parametres proches du modele deploye.
RF_MAX_ROWS = 1_500 if SMOKE_TEST else 60_000
RF_TEST_SIZE = 0.15
RF_VAL_SIZE = 0.15

LSTM_WINDOW = 60
LSTM_HORIZON = 50
LSTM_STRIDE = 2048 if SMOKE_TEST else 128
LSTM_EPOCHS = 1 if SMOKE_TEST else 30
LSTM_BATCH_SIZE = 128 if SMOKE_TEST else 64

OUTPUT_DIR = Path("models_h2s_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("RF_MAX_ROWS =", RF_MAX_ROWS)
print("LSTM_WINDOW =", LSTM_WINDOW)
print("LSTM_HORIZON =", LSTM_HORIZON)
print("LSTM_STRIDE =", LSTM_STRIDE)
print("LSTM_EPOCHS =", LSTM_EPOCHS)


## 3. Chargement du dataset

Le notebook cherche `DATASET01.csv` dans le dossier courant, dans le dossier `MODELE` fourni, puis dans le projet local. Dans Google Colab, si le fichier n'est pas trouve, il propose l'upload manuel.


In [ ]:
def find_dataset_path() -> Path:
    candidates = [
        Path("DATASET01.csv"),
        Path(r"D:\TFC_2026 (2)\Mr. RUPHIN\final\CORRECTION\MODELE\DATASET01.csv"),
        Path(r"C:\Users\labul\gas_monitoring_system\gas_monitoring_system\notebooks\DATASET01.csv"),
        Path(r"C:\Users\labul\gas_monitoring_system\gas_monitoring_system\data\DATASET01.csv"),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate

    try:
        from google.colab import files
        print("Dataset introuvable. Selectionner DATASET01.csv dans la fenetre d'upload.")
        uploaded = files.upload()
        if uploaded:
            return Path(next(iter(uploaded.keys())))
    except Exception:
        pass

    raise FileNotFoundError("DATASET01.csv introuvable. Placer le fichier dans le meme dossier que ce notebook.")

DATASET_PATH = find_dataset_path()
df_raw = pd.read_csv(DATASET_PATH)
print("Dataset utilise :", DATASET_PATH)
print("Dimensions brutes :", df_raw.shape)
print("Colonnes :", list(df_raw.columns))
display(df_raw.head())


## 4. Audit qualite avant nettoyage

Cette section verifie les colonnes attendues, les valeurs manquantes, les doublons et les valeurs invalides.


In [ ]:
required_columns = [TIME_COL, *SENSOR_COLS, TEMP_COL, HUM_COL, TRUE_CONC_COL, DATASET_LABEL_COL]
missing_columns = [col for col in required_columns if col not in df_raw.columns]
if missing_columns:
    raise ValueError(f"Colonnes manquantes : {missing_columns}")

quality_before = pd.DataFrame({
    "controle": [
        "lignes_initiales",
        "colonnes_initiales",
        "valeurs_nulles_total",
        "doublons_exacts",
        "valeurs_infinies_numeriques",
    ],
    "valeur": [
        len(df_raw),
        len(df_raw.columns),
        int(df_raw.isna().sum().sum()),
        int(df_raw.duplicated().sum()),
        int(np.isinf(df_raw.select_dtypes(include=[np.number])).sum().sum()),
    ],
})
display(quality_before)
display(df_raw[required_columns].isna().sum().to_frame("valeurs_nulles"))


## 5. Nettoyage et fusion H2S

Les quatre capteurs historiques sont convertis en numerique puis fusionnes. La variable `C_H2S_fusion_ppm` est la moyenne des capteurs H2S disponibles. Les classes utilisees par le modele sont reconstruites avec les seuils 10 ppm et 20 ppm.


In [ ]:
def classify_h2s(values):
    values = np.asarray(values, dtype=float)
    return np.select(
        [values < H2S_NORMAL_MAX, values < H2S_MODERATE_MAX],
        [0, 1],
        default=2,
    ).astype(int)

clean = df_raw[required_columns].copy()
for column in required_columns:
    clean[column] = pd.to_numeric(clean[column], errors="coerce")

rows_initial = len(clean)
clean = clean.replace([np.inf, -np.inf], np.nan).dropna().copy()
clean = clean.drop_duplicates().copy()
for sensor in SENSOR_COLS:
    clean = clean[clean[sensor] >= 0]
clean = clean[(clean[TEMP_COL] >= -40) & (clean[TEMP_COL] <= 85)]
clean = clean[(clean[HUM_COL] >= 0) & (clean[HUM_COL] <= 100)]
clean = clean.sort_values(TIME_COL).reset_index(drop=True)

clean[FUSION_COL] = clean[SENSOR_COLS].mean(axis=1)
clean[RISK_CLASS_COL] = classify_h2s(clean[FUSION_COL])
clean["risk_label"] = clean[RISK_CLASS_COL].map(RISK_LABELS)
clean["dataset_label_name"] = clean[DATASET_LABEL_COL].astype(int).map(RISK_LABELS)

label_match_rate = float((clean[RISK_CLASS_COL] == clean[DATASET_LABEL_COL].astype(int)).mean())
cleaning_report = pd.DataFrame({
    "traitement": [
        "lignes_initiales",
        "lignes_apres_nettoyage",
        "lignes_supprimees",
        "taux_correspondance_labels_dataset_vs_seuils",
    ],
    "valeur": [
        rows_initial,
        len(clean),
        rows_initial - len(clean),
        round(label_match_rate, 4),
    ],
})
display(cleaning_report)
display(clean.head())


## 6. Analyse exploratoire

On inspecte la distribution des classes et des concentrations H2S afin de comprendre l'equilibre du probleme.


In [ ]:
class_distribution = clean["risk_label"].value_counts().rename_axis("classe").reset_index(name="nombre")
class_distribution["pourcentage"] = (100 * class_distribution["nombre"] / class_distribution["nombre"].sum()).round(2)
display(class_distribution)

display(clean[[FUSION_COL, TRUE_CONC_COL, TEMP_COL, HUM_COL]].describe().round(3))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
clean[FUSION_COL].hist(ax=axes[0], bins=70, color="#0e9ec4")
axes[0].axvline(H2S_NORMAL_MAX, color="orange", linestyle="--", label="10 ppm")
axes[0].axvline(H2S_MODERATE_MAX, color="red", linestyle="--", label="20 ppm")
axes[0].set_title("Distribution H2S fusionne")
axes[0].set_xlabel("H2S (ppm)")
axes[0].legend()

class_distribution.plot(kind="bar", x="classe", y="nombre", ax=axes[1], legend=False, color="#5b8def")
axes[1].set_title("Distribution des classes")
axes[1].set_xlabel("Classe")
axes[1].set_ylabel("Nombre")

plot_sample = clean.tail(min(1500, len(clean)))
axes[2].plot(plot_sample[TIME_COL], plot_sample[FUSION_COL], color="#27ae60", linewidth=1)
axes[2].set_title("Signal H2S fusionne - fin du dataset")
axes[2].set_xlabel("Temps (s)")
axes[2].set_ylabel("H2S (ppm)")
plt.tight_layout()
plt.show()


## 7. Preparation Random Forest

Le modele Random Forest utilise `C_H2S_fusion_ppm` comme variable principale. Pour eviter que la classe normale domine, on cree un echantillon equilibre a partir d'un sous-ensemble deterministe.


In [ ]:
rf_data = clean[[FUSION_COL, RISK_CLASS_COL]].copy()
if RF_MAX_ROWS and len(rf_data) > RF_MAX_ROWS:
    sampled_parts = []
    for risk_class, group in rf_data.groupby(RISK_CLASS_COL):
        n_rows = min(len(group), max(1, RF_MAX_ROWS // 3))
        sampled_parts.append(group.sample(n_rows, random_state=SEED + int(risk_class)))
    rf_sample = pd.concat(sampled_parts, ignore_index=True).reset_index(drop=True)
else:
    rf_sample = rf_data.copy()

max_class_size = rf_sample[RISK_CLASS_COL].value_counts().max()
balanced_parts = []
for risk_class, group in rf_sample.groupby(RISK_CLASS_COL):
    balanced_parts.append(resample(group, replace=len(group) < max_class_size, n_samples=max_class_size, random_state=SEED + int(risk_class)))
rf_balanced = pd.concat(balanced_parts, ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)

X_rf = rf_balanced[[FUSION_COL]]
y_rf = rf_balanced[RISK_CLASS_COL].astype(int)

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X_rf, y_rf, test_size=RF_TEST_SIZE, random_state=SEED, stratify=y_rf
)
val_ratio = RF_VAL_SIZE / (1 - RF_TEST_SIZE)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=val_ratio, random_state=SEED, stratify=y_train_full
)

split_summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "lignes": [len(X_train), len(X_val), len(X_test)],
})
display(rf_sample[RISK_CLASS_COL].map(RISK_LABELS).value_counts().to_frame("avant_equilibrage"))
display(rf_balanced[RISK_CLASS_COL].map(RISK_LABELS).value_counts().to_frame("apres_equilibrage"))
display(split_summary)


## 8. Entrainement Random Forest

Les hyperparametres ci-dessous correspondent a une version compacte et interpretable du modele deploye.


In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=80,
    criterion="gini",
    max_depth=6,
    min_samples_split=2,
    min_samples_leaf=2,
    max_features="sqrt",
    bootstrap=True,
    oob_score=True,
    random_state=SEED,
    n_jobs=1,
)

if RUN_RF_TRAINING:
    rf_model.fit(X_train, y_train)
else:
    raise ValueError("RUN_RF_TRAINING=False : aucun modele Random Forest n'a ete charge dans ce notebook.")

print("OOB score :", round(float(rf_model.oob_score_), 4) if hasattr(rf_model, "oob_score_") else "N/A")
print("Classes :", rf_model.classes_)
print("Importance variable :", dict(zip([FUSION_COL], rf_model.feature_importances_)))


## 9. Metriques Random Forest

On calcule les metriques sur les trois partitions afin de detecter le surapprentissage et de mesurer la performance finale.


In [ ]:
def evaluate_classifier(model, X_part, y_part, split_name):
    pred = model.predict(X_part)
    metrics = {
        "split": split_name,
        "accuracy": accuracy_score(y_part, pred),
        "balanced_accuracy": balanced_accuracy_score(y_part, pred),
        "f1_macro": f1_score(y_part, pred, average="macro"),
        "f1_weighted": f1_score(y_part, pred, average="weighted"),
    }
    report = classification_report(
        y_part,
        pred,
        labels=[0, 1, 2],
        target_names=[RISK_LABELS[i] for i in [0, 1, 2]],
        output_dict=True,
        zero_division=0,
    )
    cm = confusion_matrix(y_part, pred, labels=[0, 1, 2])
    return metrics, report, cm

rf_train_metrics, rf_train_report, rf_train_cm = evaluate_classifier(rf_model, X_train, y_train, "train")
rf_val_metrics, rf_val_report, rf_val_cm = evaluate_classifier(rf_model, X_val, y_val, "validation")
rf_test_metrics, rf_test_report, rf_test_cm = evaluate_classifier(rf_model, X_test, y_test, "test")

rf_metrics_table = pd.DataFrame([rf_train_metrics, rf_val_metrics, rf_test_metrics])
display(rf_metrics_table.round(4))

test_report_df = pd.DataFrame(rf_test_report).T.round(4)
display(test_report_df)

cm_df = pd.DataFrame(
    rf_test_cm,
    index=["reel_" + RISK_LABELS[i] for i in [0, 1, 2]],
    columns=["pred_" + RISK_LABELS[i] for i in [0, 1, 2]],
)
display(cm_df)


In [ ]:
plt.figure(figsize=(5.5, 4.5))
plt.imshow(rf_test_cm, cmap="Blues")
plt.title("Matrice de confusion - Random Forest test")
plt.xticks([0, 1, 2], [RISK_LABELS[i] for i in [0, 1, 2]], rotation=25)
plt.yticks([0, 1, 2], [RISK_LABELS[i] for i in [0, 1, 2]])
for i in range(3):
    for j in range(3):
        plt.text(j, i, int(rf_test_cm[i, j]), ha="center", va="center", color="black")
plt.xlabel("Prediction")
plt.ylabel("Classe reelle")
plt.tight_layout()
plt.show()

importance = pd.Series(rf_model.feature_importances_, index=[FUSION_COL]).sort_values(ascending=False)
display(importance.to_frame("importance"))


## 10. Courbe d'apprentissage Random Forest

Cette cellule peut etre longue sur machine faible. Elle est desactivee automatiquement en mode test rapide.


In [ ]:
if RUN_LEARNING_CURVE:
    sizes, train_scores, val_scores = learning_curve(
        rf_model,
        X_train_full,
        y_train_full,
        cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED),
        scoring="f1_macro",
        train_sizes=np.linspace(0.15, 1.0, 6),
        n_jobs=1,
    )
    learning_df = pd.DataFrame({
        "train_size": sizes,
        "train_f1_macro": train_scores.mean(axis=1),
        "validation_f1_macro": val_scores.mean(axis=1),
    })
    display(learning_df.round(4))
    plt.figure(figsize=(7, 4))
    plt.plot(learning_df["train_size"], learning_df["train_f1_macro"], marker="o", label="train")
    plt.plot(learning_df["train_size"], learning_df["validation_f1_macro"], marker="o", label="validation")
    plt.title("Courbe d'apprentissage Random Forest")
    plt.xlabel("Nombre d'exemples d'entrainement")
    plt.ylabel("F1 macro")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    learning_df = pd.DataFrame()
    print("Courbe d'apprentissage ignoree en mode test rapide.")


## 11. Preparation LSTM

Le LSTM recoit une fenetre temporelle de 60 mesures et predit la concentration H2S future apres 50 pas. Le `stride` permet de reduire le nombre de sequences tout en conservant la chronologie.


In [ ]:
lstm_source = clean[[TIME_COL, FUSION_COL]].sort_values(TIME_COL).reset_index(drop=True).copy()
lstm_source["future_h2s"] = lstm_source[FUSION_COL].shift(-LSTM_HORIZON)
lstm_source = lstm_source.dropna().reset_index(drop=True)

train_end_idx = int(len(lstm_source) * 0.70)
val_end_idx = int(len(lstm_source) * 0.85)
train_min = float(lstm_source.loc[:train_end_idx, FUSION_COL].min())
train_max = float(lstm_source.loc[:train_end_idx, FUSION_COL].max())
if train_max <= train_min:
    train_max = train_min + 1.0


def scale_h2s(values):
    return np.clip((np.asarray(values, dtype=float) - train_min) / (train_max - train_min), 0, 1)


def unscale_h2s(values):
    return np.asarray(values, dtype=float) * (train_max - train_min) + train_min


def make_sequences(frame, window=LSTM_WINDOW, stride=LSTM_STRIDE):
    h2s_scaled = scale_h2s(frame[FUSION_COL].values)
    target_scaled = scale_h2s(frame["future_h2s"].values)
    X_seq, y_seq = [], []
    last_start = len(frame) - window
    for start in range(0, max(0, last_start), stride):
        end = start + window
        X_seq.append(h2s_scaled[start:end])
        y_seq.append(target_scaled[end - 1])
    X_seq = np.asarray(X_seq, dtype="float32").reshape(-1, window, 1)
    y_seq = np.asarray(y_seq, dtype="float32").reshape(-1, 1)
    return X_seq, y_seq

train_part = lstm_source.iloc[:train_end_idx].copy()
val_part = lstm_source.iloc[train_end_idx:val_end_idx].copy()
test_part = lstm_source.iloc[val_end_idx:].copy()

X_lstm_train, y_lstm_train = make_sequences(train_part)
X_lstm_val, y_lstm_val = make_sequences(val_part)
X_lstm_test, y_lstm_test = make_sequences(test_part)

lstm_split_table = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "lignes_source": [len(train_part), len(val_part), len(test_part)],
    "sequences": [len(X_lstm_train), len(X_lstm_val), len(X_lstm_test)],
})
print("Normalisation min train :", train_min)
print("Normalisation max train :", train_max)
display(lstm_split_table)
print("Shape X train :", X_lstm_train.shape)


## 12. Construction et entrainement LSTM

Architecture compacte compatible avec le modele deploye : LSTM(12), Dropout(0.15), Dense(8), Dense(1).


In [ ]:
def build_lstm_model(window=LSTM_WINDOW):
    if tf is None:
        raise ImportError("TensorFlow est requis pour entrainer le LSTM.")
    model = Sequential([
        Input(shape=(window, 1)),
        LSTM(12, activation="tanh"),
        Dropout(0.15),
        Dense(8, activation="relu"),
        Dense(1, activation="linear"),
    ])
    model.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return model

if RUN_LSTM_TRAINING:
    if len(X_lstm_train) == 0 or len(X_lstm_val) == 0 or len(X_lstm_test) == 0:
        raise ValueError("Pas assez de sequences LSTM. Reduire LSTM_WINDOW, LSTM_HORIZON ou LSTM_STRIDE.")
    lstm_model = build_lstm_model()
    callbacks_list = [
        EarlyStopping(monitor="val_loss", patience=max(3, LSTM_EPOCHS), restore_best_weights=True, min_delta=1e-5),
        ReduceLROnPlateau(monitor="val_loss", patience=4, factor=0.5, min_lr=1e-5),
    ]
    history = lstm_model.fit(
        X_lstm_train,
        y_lstm_train,
        validation_data=(X_lstm_val, y_lstm_val),
        epochs=LSTM_EPOCHS,
        batch_size=LSTM_BATCH_SIZE,
        callbacks=callbacks_list,
        verbose=1,
    )
else:
    raise ValueError("RUN_LSTM_TRAINING=False : aucun modele LSTM n'a ete charge dans ce notebook.")

lstm_model.summary()


## 13. Courbes d'entrainement LSTM


In [ ]:
history_df = pd.DataFrame(history.history)
display(history_df.tail())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
history_df[["loss", "val_loss"]].plot(ax=axes[0])
axes[0].set_title("Loss LSTM")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE normalise")
history_df[["mae", "val_mae"]].plot(ax=axes[1])
axes[1].set_title("MAE LSTM")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("MAE normalisee")
plt.tight_layout()
plt.show()


## 14. Metriques LSTM

Les predictions sont reconverties en ppm pour calculer MAE, RMSE et R2 dans l'unite du gaz.


In [ ]:
y_pred_scaled = lstm_model.predict(X_lstm_test, batch_size=1024).reshape(-1)
y_true_scaled = y_lstm_test.reshape(-1)

y_pred_ppm = np.clip(unscale_h2s(y_pred_scaled), 0, None)
y_true_ppm = unscale_h2s(y_true_scaled)

lstm_metrics = {
    "mae_ppm": mean_absolute_error(y_true_ppm, y_pred_ppm),
    "rmse_ppm": float(np.sqrt(mean_squared_error(y_true_ppm, y_pred_ppm))),
    "r2": r2_score(y_true_ppm, y_pred_ppm) if len(y_true_ppm) > 1 else np.nan,
    "test_sequences": len(y_true_ppm),
}
display(pd.Series(lstm_metrics).to_frame("valeur"))

lstm_predictions_df = pd.DataFrame({
    "h2s_reel_ppm": y_true_ppm,
    "h2s_predit_ppm": y_pred_ppm,
    "erreur_ppm": y_pred_ppm - y_true_ppm,
})
display(lstm_predictions_df.head(20).round(3))

plt.figure(figsize=(12, 4))
plot_n = min(250, len(lstm_predictions_df))
plt.plot(lstm_predictions_df["h2s_reel_ppm"].iloc[:plot_n].values, label="H2S reel", color="#2f80ed")
plt.plot(lstm_predictions_df["h2s_predit_ppm"].iloc[:plot_n].values, label="H2S predit", color="#eb5757", linestyle="--")
plt.title("LSTM - H2S reel vs predit")
plt.xlabel("Sequence de test")
plt.ylabel("H2S (ppm)")
plt.legend()
plt.tight_layout()
plt.show()


## 15. Classification du risque futur

La valeur future predite par le LSTM est transformee en classe de danger avec les memes seuils que le dashboard.


In [ ]:
future_classes = classify_h2s(y_pred_ppm)
future_distribution = pd.Series(future_classes).map(RISK_LABELS).value_counts().rename_axis("classe_future").reset_index(name="nombre")
future_distribution["pourcentage"] = (100 * future_distribution["nombre"] / future_distribution["nombre"].sum()).round(2)
display(future_distribution)

future_rf_features = pd.DataFrame({FUSION_COL: y_pred_ppm})
future_rf_classes = rf_model.predict(future_rf_features)
future_rf_distribution = pd.Series(future_rf_classes).map(RISK_LABELS).value_counts().rename_axis("classe_future_rf").reset_index(name="nombre")
future_rf_distribution["pourcentage"] = (100 * future_rf_distribution["nombre"] / future_rf_distribution["nombre"].sum()).round(2)
display(future_rf_distribution)


## 16. Synthese finale des metriques

Cette cellule regroupe les resultats principaux pour le rapport.


In [ ]:
summary_metrics = {
    "dataset_path": str(DATASET_PATH),
    "rows_clean": int(len(clean)),
    "label_match_rate_with_dataset_labels": label_match_rate,
    "rf_oob_score": float(rf_model.oob_score_) if hasattr(rf_model, "oob_score_") else None,
    "rf_test_accuracy": rf_test_metrics["accuracy"],
    "rf_test_balanced_accuracy": rf_test_metrics["balanced_accuracy"],
    "rf_test_f1_macro": rf_test_metrics["f1_macro"],
    "lstm_window": LSTM_WINDOW,
    "lstm_horizon": LSTM_HORIZON,
    "lstm_stride": LSTM_STRIDE,
    "lstm_epochs_executed": len(history_df),
    "lstm_test_mae_ppm": lstm_metrics["mae_ppm"],
    "lstm_test_rmse_ppm": lstm_metrics["rmse_ppm"],
    "lstm_test_r2": lstm_metrics["r2"],
}
summary_df = pd.Series(summary_metrics, name="valeur").to_frame()
display(summary_df)


## 17. Sauvegarde optionnelle des modeles et metadonnees

Par defaut, la sauvegarde est desactivee pour eviter d'ecraser un modele valide. Mettre `SAVE_MODELS=True` apres validation des metriques.


In [ ]:
metadata = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "dataset_path": str(DATASET_PATH),
    "rows_clean": int(len(clean)),
    "sensor_columns": SENSOR_COLS,
    "fusion_formula": "C_H2S_fusion_ppm = mean(Sensor1, Sensor2, Sensor3, Sensor4)",
    "deployment_note": "Dans le prototype a capteur unique MQ-136, C_H2S_fusion_ppm = h2s_ppm.",
    "gas": "H2S",
    "co_used": False,
    "thresholds": {
        "NORMAL": "H2S < 10 ppm",
        "MODERE": "10 <= H2S < 20 ppm",
        "DANGEREUX": "H2S >= 20 ppm",
    },
    "cleaning_report": cleaning_report.to_dict(orient="records"),
    "class_distribution": class_distribution.to_dict(orient="records"),
    "rf_metrics": {
        "train": rf_train_metrics,
        "validation": rf_val_metrics,
        "test": rf_test_metrics,
        "test_confusion_matrix": rf_test_cm.tolist(),
        "test_classification_report": rf_test_report,
        "feature_importance": {FUSION_COL: float(rf_model.feature_importances_[0])},
    },
    "lstm_metrics": summary_metrics,
}

if SAVE_MODELS:
    rf_path = OUTPUT_DIR / "random_forest_h2s_fusion.pkl"
    lstm_path = OUTPUT_DIR / "lstm_h2s_fusion.h5"
    metadata_path = OUTPUT_DIR / "metadata_h2s_fusion.json"
    joblib.dump(rf_model, rf_path)
    lstm_model.save(lstm_path)
    metadata_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Modeles sauvegardes :")
    print(rf_path)
    print(lstm_path)
    print(metadata_path)
else:
    print("SAVE_MODELS=False : aucun fichier modele n'a ete ecrase.")


## 18. Interpretation pour le rapport

Points a commenter dans le travail :

1. Le Random Forest classe le danger instantane a partir de la concentration H2S fusionnee.
2. Les tres bons scores Random Forest sont attendus parce que les classes sont definies par des seuils H2S nets.
3. Le LSTM est plus difficile a optimiser, car il predit une valeur future continue.
4. La MAE et la RMSE du LSTM doivent etre interpretees en ppm.
5. Pour le prototype actuel, l'entree terrain est un seul capteur MQ-136 ; la fusion historique est donc remplacee par `h2s_ppm`.
6. `co_ppm` n'est pas utilise dans ce modele.
